<div style="background-color:#000;"><img src="pqn.png"></img></div><div><a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.</div>

## Library installation

This installs all the libraries needed to fetch market data, compute risk metrics, and plot the results.

In [ ]:
!pip install yfinance pandas numpy matplotlib

## Imports and setup

numpy handles the numerical operations like percentiles and dot products. pandas structures our return data as DataFrames. yfinance downloads historical stock prices from Yahoo Finance. matplotlib produces the histogram that visualizes VaR and CVaR.

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

## Download and prepare portfolio returns

We define a universe of roughly 90 large-cap U.S. stocks, similar to the S&P 100 index, to simulate a diversified equity portfolio.

In [ ]:
oex = [
    'MMM', 'T', 'ABBV', 'ABT', 'ACN', 'ALL', 'GOOGL', 'GOOG',
    'MO', 'AMZN', 'AXP', 'AIG', 'AMGN', 'AAPL', 'BAC',
    'BRK-B', 'BIIB', 'BLK', 'BA', 'BMY', 'CVS', 'COF', 'CAT',
    'CVX', 'CSCO', 'C', 'KO', 'CL', 'CMCSA', 'COP', 'DHR',
    'DUK', 'DD', 'EMC', 'EMR', 'EXC', 'XOM', 'META', 'FDX',
    'F', 'GD', 'GE', 'GM', 'GILD', 'GS', 'HAL', 'HD', 'HON',
    'INTC', 'IBM', 'JPM', 'JNJ', 'KMI', 'LLY', 'LMT', 'LOW',
    'MA', 'MCD', 'MDT', 'MRK', 'MET', 'MSFT', 'MS', 'NKE',
    'NEE', 'OXY', 'ORCL', 'PYPL', 'PEP', 'PFE', 'PM', 'PG',
    'QCOM', 'SLB', 'SPG', 'SO', 'SBUX', 'TGT', 'TXN', 'BK',
    'USB', 'UNP', 'UPS', 'UNH', 'VZ', 'V', 'WMT', 'WBA',
    'DIS', 'WFC',
]

In [ ]:
num_stocks = len(oex)

yfinance downloads adjusted closing prices for all stocks in a single call, returning a multi-level DataFrame that covers roughly two years of daily data.

In [ ]:
data = yf.download(oex, start='2014-01-01', end='2016-04-04')

We compute daily percentage returns and then subtract each stock's mean return to center the distribution around zero. Demeaning isolates the risk component of returns, which is what VaR and CVaR are designed to measure.

In [ ]:
returns = data.Close.pct_change()
returns = returns - returns.mean(skipna=True)

Centering returns is a common preprocessing step in risk analysis. It removes the drift so that our loss estimates reflect pure volatility and tail behavior rather than being skewed by average gains or losses over the sample period.

## Generate random portfolio weights

The scale function normalizes a vector so its absolute values sum to one, producing valid portfolio weights. We generate random weights to simulate an arbitrary allocation across all stocks.

In [ ]:
def scale(x):
    return x / np.sum(np.abs(x))

In [ ]:
weights = scale(np.random.random(num_stocks))

In [ ]:
plt.bar(np.arange(num_stocks), weights)
plt.show()

Using random weights lets us focus on the risk measurement technique itself rather than on portfolio construction. In practice, you would replace these with your actual target allocations. The bar chart gives a quick visual check that no single stock dominates the portfolio.

## Compute Value at Risk and CVaR

This function computes historical VaR by taking the weighted sum of individual stock returns over a lookback window and finding the loss threshold at the given confidence level.

In [ ]:
def value_at_risk(
    value_invested,
    returns,
    weights,
    alpha=0.95,
    lookback_days=500,
):
    returns = returns.fillna(0.0)
    portfolio_returns = returns.iloc[-lookback_days:].dot(weights)
    return (
        np.percentile(portfolio_returns, 100 * (1 - alpha))
        * value_invested
    )

The dot product of daily returns and weights produces a single portfolio return for each day. numpy's percentile function then finds the cutoff below which only the worst 5% of days fall. Multiplying by the invested amount converts that percentage into a dollar loss, which is the VaR figure.

This function computes CVaR by first calling value_at_risk to find the threshold, then averaging only the portfolio returns that fell below that threshold.

In [ ]:
def cvar(
    value_invested,
    returns,
    weights,
    alpha=0.95,
    lookback_days=500,
):
    var = value_at_risk(
        value_invested, returns, weights, alpha,
        lookback_days=lookback_days,
    )
    returns = returns.fillna(0.0)
    portfolio_returns = returns.iloc[-lookback_days:].dot(weights)
    var_pct_loss = var / value_invested
    return (
        np.nanmean(
            portfolio_returns[portfolio_returns < var_pct_loss]
        )
        * value_invested
    )

CVaR answers the question VaR leaves open: when losses exceed the threshold, how bad do they actually get on average? This is why regulators under Basel III now require expected shortfall instead of VaR. By averaging the tail losses rather than just marking their boundary, CVaR penalizes portfolios with fat tails and gives a more honest picture of worst-case exposure.

We assume a $1,000,000 portfolio and print both risk measures so we can compare the dollar amounts directly.

In [ ]:
value_invested = 1000000

In [ ]:
print(cvar(value_invested, returns, weights))
print(value_at_risk(value_invested, returns, weights))

The CVaR number will always be larger in magnitude than VaR because it averages the losses beyond the VaR cutoff. The gap between the two numbers tells you how much additional pain hides in the tail. A small gap suggests the tail is thin and well-behaved, while a large gap signals that extreme days are significantly worse than the threshold day.

## Visualize the tail risk distribution

We reconstruct the portfolio return series and convert both risk measures to return percentages so we can overlay them on a histogram of daily outcomes.

In [ ]:
lookback_days = 500

In [ ]:
portfolio_returns = (
    returns.fillna(0.0).iloc[-lookback_days:].dot(weights)
)

In [ ]:
portfolio_VaR = value_at_risk(value_invested, returns, weights)
portfolio_VaR_return = portfolio_VaR / value_invested

In [ ]:
portfolio_CVaR = cvar(value_invested, returns, weights)
portfolio_CVaR_return = portfolio_CVaR / value_invested

The histogram splits returns into two groups: days above the VaR threshold and days below it. The solid red line marks VaR and the dashed red line marks CVaR, showing exactly where each measure sits relative to the distribution of actual losses.

In [ ]:
plt.hist(
    portfolio_returns[portfolio_returns > portfolio_VaR_return],
    bins=20,
)
plt.hist(
    portfolio_returns[portfolio_returns < portfolio_VaR_return],
    bins=10,
)
plt.axvline(portfolio_VaR_return, color='red', linestyle='solid')
plt.axvline(portfolio_CVaR_return, color='red', linestyle='dashed')
plt.legend(['VaR', 'CVaR', 'Returns', 'Returns < VaR'])
plt.title('Historical VaR and CVaR')
plt.xlabel('Return')
plt.ylabel('Observation Frequency')
plt.show()

This chart is the core deliverable of the analysis. The colored bars to the left of the solid red line represent the worst days in your portfolio's history. VaR only tells you where that region starts, but the dashed CVaR line shows the average position within it. When you build your own risk reports, this visualization makes it immediately clear whether your tail risk is concentrated near the VaR boundary or spread out into much deeper losses.

<a href="https://pyquantnews.com/">PyQuant News</a> is where finance practitioners level up with Python for quant finance, algorithmic trading, and market data analysis. Looking to get started? Check out the fastest growing, top-selling course to <a href="https://www.pyquantnews.com/getting-started-with-python-for-quant-finance/">get started with Python for quant finance</a>. For educational purposes. Not investment advice. Use at your own risk.